# Part 13: Macro Indicators for Market Timing



In [1]:
SW_IND_MAP_1 = {
    '801010.SI': '农林牧渔',
    '801030.SI': '基础化工',
    '801040.SI': '钢铁',
    '801050.SI': '有色金属',
    '801080.SI': '电子',
    '801110.SI': '家用电器',
    '801120.SI': '食品饮料',
    '801130.SI': '纺织服饰',
    '801140.SI': '轻工制造',
    '801150.SI': '医药生物',
    '801160.SI': '公用事业',
    '801170.SI': '交通运输',
    '801180.SI': '房地产',
    '801200.SI': '商贸零售',
    '801210.SI': '社会服务',
    '801230.SI': '综合',
    '801710.SI': '建筑材料',
    '801720.SI': '建筑装饰',
    '801730.SI': '电力设备',
    '801740.SI': '国防军工',
    '801750.SI': '计算机',
    '801760.SI': '传媒',
    '801770.SI': '通信',
    '801780.SI': '银行',
    '801790.SI': '非银金融',
    '801880.SI': '汽车',
    '801890.SI': '机械设备',
} 
#     '801950.SI': '煤炭',
#     '801960.SI': '石油石化',
#     '801970.SI': '环保',
#     '801980.SI': '美容护理'
# }

In [2]:
import pandas as pd
import numpy as np
import math
import pickle
from scipy.stats import gmean
from scipy.stats import percentileofscore
import dateutil.relativedelta


import warnings
warnings.filterwarnings('ignore')

df_month = pd.read_excel('经济增长_month.xlsx')
df_day = pd.read_excel('经济增长_day.xlsx')


In [3]:
df_day

,date,中国:中债国债到期收益率:10年,中债国债到期收益率:1个月
0,2011-01-04,3.8606,2.7181
1,2011-01-05,3.8132,2.4791
2,2011-01-06,3.8339,2.3000
3,2011-01-07,3.8237,2.2689
4,2011-01-10,3.8449,2.2408
...,...,...,...
3098,2023-05-25,2.7065,1.5924
3099,2023-05-26,2.7205,1.5777
3100,2023-05-29,2.6950,1.5723
3101,2023-05-30,2.7000,1.5605


In [4]:
df_day['国债利差 10Y-1M'] = df_day['中国:中债国债到期收益率:10年'] - df_day['中债国债到期收益率:1个月']
df_day.set_index('date', inplace=True)
df_day

,中国:中债国债到期收益率:10年,中债国债到期收益率:1个月,国债利差 10Y-1M
date,,,
2011-01-04,3.8606,2.7181,1.1425
2011-01-05,3.8132,2.4791,1.3341
2011-01-06,3.8339,2.3000,1.5339
2011-01-07,3.8237,2.2689,1.5548
2011-01-10,3.8449,2.2408,1.6041
...,...,...,...
2023-05-25,2.7065,1.5924,1.1141
2023-05-26,2.7205,1.5777,1.1428
2023-05-29,2.6950,1.5723,1.1227


In [5]:
df_month

,date,中国:PMI:新出口订单,中国:产量:发电量:当月值,中国:金融机构:中长期贷款余额,中国:社会融资规模:新增人民币贷款:当月值
0,2011-01-31,50.7,3672.2746,295103.41,10263.0
1,2011-02-28,50.9,3100.8000,298789.94,5377.0
2,2011-03-31,52.5,3830.1000,302720.88,6794.0
3,2011-04-30,51.3,3663.8000,306483.51,7430.0
4,2011-05-31,51.1,3775.4000,308931.15,5516.0
...,...,...,...,...,...
144,2023-01-31,46.1,0.0000,1461844.06,49314.0
145,2023-02-28,52.4,0.0000,1473835.11,18184.0
146,2023-03-31,50.4,7172.9246,1500791.77,39487.0
147,2023-04-30,47.6,6583.5000,1506304.81,4431.0


In [6]:
df_month.set_index('date', inplace=True)
df_month

,中国:PMI:新出口订单,中国:产量:发电量:当月值,中国:金融机构:中长期贷款余额,中国:社会融资规模:新增人民币贷款:当月值
date,,,,
2011-01-31,50.7,3672.2746,295103.41,10263.0
2011-02-28,50.9,3100.8000,298789.94,5377.0
2011-03-31,52.5,3830.1000,302720.88,6794.0
2011-04-30,51.3,3663.8000,306483.51,7430.0
2011-05-31,51.1,3775.4000,308931.15,5516.0
...,...,...,...,...
2023-01-31,46.1,0.0000,1461844.06,49314.0
2023-02-28,52.4,0.0000,1473835.11,18184.0
2023-03-31,50.4,7172.9246,1500791.77,39487.0


In [7]:
# window_sizes = [60,72,84]
window_sizes = [60]

In [9]:
def calculate_exceeded_median(data, column_name, window_size):
    rolling_median = data[column_name].rolling(window=window_size).median()
    exceeded_median = (data[column_name] > rolling_median).astype(int)
    return exceeded_median

In [10]:
column_names = ['中国:PMI:新出口订单', '中国:产量:发电量:当月值', '中国:社会融资规模:新增人民币贷款:当月值', '中国:金融机构:中长期贷款余额']

for hhwindow_size in window_sizes:

    # 对每一列执行相同的功能
    for column_name in column_names:
        # 计算滚动区间中的中位数并标记超出中位数的情况
        exceeded_median = calculate_exceeded_median(df_month, column_name, hhwindow_size)
        df_month[column_name + " signal" + str(hhwindow_size)] = exceeded_median

    df_day['国债利差 10Y-1M signal' + str(hhwindow_size)] =  calculate_exceeded_median(df_day, '国债利差 10Y-1M', hhwindow_size)
    


In [11]:
df_month

,中国:PMI:新出口订单,中国:产量:发电量:当月值,中国:金融机构:中长期贷款余额,中国:社会融资规模:新增人民币贷款:当月值,中国:PMI:新出口订单 signal60,中国:产量:发电量:当月值 signal60,中国:社会融资规模:新增人民币贷款:当月值 signal60,中国:金融机构:中长期贷款余额 signal60
date,,,,,,,,
2011-01-31,50.7,3672.2746,295103.41,10263.0,0,0,0,0
2011-02-28,50.9,3100.8000,298789.94,5377.0,0,0,0,0
2011-03-31,52.5,3830.1000,302720.88,6794.0,0,0,0,0
2011-04-30,51.3,3663.8000,306483.51,7430.0,0,0,0,0
2011-05-31,51.1,3775.4000,308931.15,5516.0,0,0,0,0
...,...,...,...,...,...,...,...,...
2023-01-31,46.1,0.0000,1461844.06,49314.0,0,0,1,1
2023-02-28,52.4,0.0000,1473835.11,18184.0,1,0,1,1
2023-03-31,50.4,7172.9246,1500791.77,39487.0,1,1,1,1


In [39]:
df_day

,中国:中债国债到期收益率:10年,中债国债到期收益率:1个月,国债利差 10Y-1M,国债利差 10Y-1M signal60
date,,,,
2011-01-04,3.8606,2.7181,1.1425,0
2011-01-05,3.8132,2.4791,1.3341,0
2011-01-06,3.8339,2.3000,1.5339,0
2011-01-07,3.8237,2.2689,1.5548,0
2011-01-10,3.8449,2.2408,1.6041,0
...,...,...,...,...
2023-05-25,2.7065,1.5924,1.1141,1
2023-05-26,2.7205,1.5777,1.1428,1
2023-05-29,2.6950,1.5723,1.1227,1


In [40]:
df_month.to_excel('月度数据触发.xlsx')
df_day.to_excel('日度数据触发.xlsx')

In [12]:
def win_rate_test(sig_df, rets_df, window_size):

    sig_df = sig_df[(sig_df.index < rets_df.index.max())]

    record = []
    for win in window_size:
        total = sig_df[:len(sig_df.index)-win+1].sum()
        win_num = 0
        for i in range(len(sig_df.index)-win+1):

            # 事件发生的时间
            date = sig_df.index[i]
            
            # 如果发生了事情
            if sig_df.loc[date] == 1:
                date = pd.to_datetime(date, format='%Y%m%d')
                # print(date)
                # print(rets_df.loc[date:][0])
                # print(date + window)
                # print(rets_df.loc[date:][0].index)
                # break
            
                if rets_df.loc[date:][0] < rets_df.loc[date:][win]:
                    win_num += 1

        
        win_ratio = win_num / total
        # print('{}日胜率={}, 触发次数={}'.format(win, win_ratio, total))      
        record.append(win_ratio)
    
    record.append(total)
    result = pd.DataFrame(record, index=['20日胜率', '60日胜率','触发次数'])
    result = result.T
    return result

In [13]:
start_time = pd.to_datetime('20110101', format='%Y%m%d')

closeprice = pd.read_pickle('IndexQuote_ClosePrice.txt')
closeprice.index = pd.to_datetime(closeprice.index)

closeprice_500 = closeprice['000905.SH'].loc[closeprice['000905.SH'].index >= start_time]
closeprice_cyb = closeprice['399006.SZ'].loc[closeprice['399006.SZ'].index >= start_time]
closeprice_300 = closeprice['000300.SH'].loc[closeprice['000300.SH'].index >= start_time]

close_price_行业 = pd.read_pickle("IndexQuote_SWS_ClosePrice.txt")
close_price_行业.index = pd.to_datetime(close_price_行业.index)


close_price_行业 = close_price_行业.loc[close_price_行业.index >= start_time]
close_price_行业 = close_price_行业[SW_IND_MAP_1.keys()]


In [14]:
closeprice_500

2011-01-03    4936.7160
2011-01-04    5011.4980
2011-01-05    5028.7660
2011-01-06    5001.2830
2011-01-07    4975.1440
                ...    
2023-01-18    6151.4557
2023-01-19    6204.3261
2023-01-20    6251.4337
2023-01-30    6283.3272
2023-01-31    6289.1500
Name: 000905.SH, Length: 3348, dtype: float64

In [15]:
result_dict = {}
# 对每一列执行相同的功能
for column_name in column_names:
    # 计算滚动区间中的中位数并标记超出中位数的情况
    exceeded_median = calculate_exceeded_median(df_month, column_name, hhwindow_size)
    result_dict[column_name] = exceeded_median

# 计算国债利差
result_dict['国债利差 10Y-1M'] =  calculate_exceeded_median(df_day, '国债利差 10Y-1M', hhwindow_size)
result_dict

{'中国:PMI:新出口订单': date
 2011-01-31    0
 2011-02-28    0
 2011-03-31    0
 2011-04-30    0
 2011-05-31    0
              ..
 2023-01-31    0
 2023-02-28    1
 2023-03-31    1
 2023-04-30    0
 2023-05-31    0
 Name: 中国:PMI:新出口订单, Length: 149, dtype: int64,
 '中国:产量:发电量:当月值': date
 2011-01-31    0
 2011-02-28    0
 2011-03-31    0
 2011-04-30    0
 2011-05-31    0
              ..
 2023-01-31    0
 2023-02-28    0
 2023-03-31    1
 2023-04-30    1
 2023-05-31    1
 Name: 中国:产量:发电量:当月值, Length: 149, dtype: int64,
 '中国:社会融资规模:新增人民币贷款:当月值': date
 2011-01-31    0
 2011-02-28    0
 2011-03-31    0
 2011-04-30    0
 2011-05-31    0
              ..
 2023-01-31    1
 2023-02-28    1
 2023-03-31    1
 2023-04-30    0
 2023-05-31    0
 Name: 中国:社会融资规模:新增人民币贷款:当月值, Length: 149, dtype: int64,
 '中国:金融机构:中长期贷款余额': date
 2011-01-31    0
 2011-02-28    0
 2011-03-31    0
 2011-04-30    0
 2011-05-31    0
              ..
 2023-01-31    1
 2023-02-28    1
 2023-03-31    1
 2023-04-30    1
 2023-05-31   

In [16]:
for hhwindow_size in window_sizes:

    window_size = [20,60]

    output_dict = {}  # Dictionary to store the results
    for name,item in result_dict.items():
        # print()
        # print(name)
        # # print(item)
        # print(len(item))
        temp_500 = win_rate_test(item, closeprice_500, window_size)
        temp_300 = win_rate_test(item, closeprice_300, window_size)
        temp_cyb = win_rate_test(item, closeprice_cyb, window_size)

# temp_500
# temp_300

In [50]:
result_dict

{'中国:PMI:新出口订单': date
 2011-01-31    0
 2011-02-28    0
 2011-03-31    0
 2011-04-30    0
 2011-05-31    0
              ..
 2023-01-31    0
 2023-02-28    1
 2023-03-31    1
 2023-04-30    0
 2023-05-31    0
 Name: 中国:PMI:新出口订单, Length: 149, dtype: int64,
 '中国:产量:发电量:当月值': date
 2011-01-31    0
 2011-02-28    0
 2011-03-31    0
 2011-04-30    0
 2011-05-31    0
              ..
 2023-01-31    0
 2023-02-28    0
 2023-03-31    1
 2023-04-30    1
 2023-05-31    1
 Name: 中国:产量:发电量:当月值, Length: 149, dtype: int64,
 '中国:社会融资规模:新增人民币贷款:当月值': date
 2011-01-31    0
 2011-02-28    0
 2011-03-31    0
 2011-04-30    0
 2011-05-31    0
              ..
 2023-01-31    1
 2023-02-28    1
 2023-03-31    1
 2023-04-30    0
 2023-05-31    0
 Name: 中国:社会融资规模:新增人民币贷款:当月值, Length: 149, dtype: int64,
 '中国:金融机构:中长期贷款余额': date
 2011-01-31    0
 2011-02-28    0
 2011-03-31    0
 2011-04-30    0
 2011-05-31    0
              ..
 2023-01-31    1
 2023-02-28    1
 2023-03-31    1
 2023-04-30    1
 2023-05-31   

In [17]:
hhwindow_size = 60

window_size = [20,60]

output_dict = {}  # Dictionary to store the results
for name,item in result_dict.items():

    temp_500 = win_rate_test(item, closeprice_500, window_size)
    temp_300 = win_rate_test(item, closeprice_300, window_size)
    temp_cyb = win_rate_test(item, closeprice_cyb, window_size)

    industry_results = {}
    
    for i, j in close_price_行业.items():

        industry_name = SW_IND_MAP_1[i]
        # print(industry_name)
        temp = win_rate_test(item, j, window_size)
        industry_results[industry_name] = temp

    output_dict[name] = {
        '沪深500指数': temp_500,
        '沪深300指数': temp_300,
        '创业板指数': temp_cyb,
        '行业': industry_results
    }

In [18]:
output_df = pd.DataFrame(columns=['因子', '标的', '20日胜率', '60日胜率', '触发次数','行业'])

for factor_name, factor_result in output_dict.items():
    for stock_name, stock_result in factor_result.items():
        if isinstance(stock_result, pd.DataFrame):
            temp_df = stock_result.reset_index()
            temp_df['因子'] = factor_name
            temp_df['标的'] = stock_name
            # output_df = output_df.append(temp_df, ignore_index=True)
            output_df = pd.concat([output_df, temp_df], ignore_index=True)
        elif isinstance(stock_result, dict):
            for industry_name, industry_result in stock_result.items():
                temp_df = industry_result.reset_index()
                temp_df['因子'] = factor_name
                temp_df['标的'] = stock_name
                temp_df['行业'] = industry_name
                # output_df = output_df.append(temp_df, ignore_index=True)
                output_df = pd.concat([output_df, temp_df], ignore_index=True)

# Save the output_df DataFrame to an Excel file
output_df.to_excel(f"{hhwindow_size}_results.xlsx")